In [ ]:
#Import the necessary libraries
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.preprocessing import StandardScaler
from scipy.stats import qmc
import numpy as np
from scipy.optimize import minimize

In [ ]:
X = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939, 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003, 0.41363143, 0.58523563],
    [0.63021764, 0.8380969, 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408, 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454, 0.6354384, 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009, 0.69156848, 0.6555429],
    [0.17597754, 0.6244165, 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952, 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983, 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764, 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547, 0.07966402],
    [0.057895, 0.491672, 0.247422, 0.218118, 0.420428, 0.730969],
    [0.861674, 0.710584, 0.916207, 0.413415, 0.568112, 0.688380],
    [0.00754461, 0.05604467, 0.76901999, 0.05610508, 0.37627984, 0.80855666],
    [0.410355, 0.404626, 0.415739, 0.605124, 0.360664, 0.916164],
    [0.408786, 0.272011, 0.015960, 0.913927, 0.779885, 0.884180],
    [0.363794, 0.381721, 0.246385, 0.582342, 0.305840, 0.924433],
    [0.033969, 0.126740, 0.514117, 0.133496, 0.374088, 0.621625],
    [0.027231, 0.085405, 0.752894, 0.120046, 0.322614, 0.589848],
    [0.232502, 0.129287, 0.909833, 0.068397, 0.394889, 0.611060],
    [0.024767, 0.666722, 0.492560, 0.088079, 0.376655, 0.489081],
    [0.033972, 0.028757, 0.450864, 0.057456, 0.268097, 0.677730],
    [0.067142, 0.084164, 0.391241, 0.179196, 0.379401, 0.563393]

])

y = np.array([
    0.6044327, 0.56275307, 0.00750324, 0.0614243, 0.2730468, 0.08374657,
    1.3649683, 0.09264495, 0.0178696, 0.03356494, 0.0735163, 0.2063097,
    0.00882563, 0.26840032, 0.61152553, 0.01479818, 0.27489251, 0.06676325,
    0.04211835, 0.00270147, 0.01820907, 0.00701603, 0.10050661, 0.47539552,
    0.67514163, 0.51645722, 0.00377748, 0.00313433, 0.02134252, 0.09541116,
    1.36497012019043, 0.0655047775737983, 1.12760535430941, 0.507529317551999,
    0.0007165677147890787, 0.5763159658663746, 2.402348648054967, 1.7273032249712312,
    1.0330808540321927, 0.7256545337861763, 1.8200116920477367, 2.3961650586254293
])

print(X.shape)
print(y.shape)

In [ ]:
# scale the data
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
print(f"\nAfter scaling:")
print(f"X_scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")
print(f"y_scaled range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")

#GP setup
kernel = ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3)) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    length_scale_bounds=(0.01, 1000.0)
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=30,  # More restarts
    alpha=0.01,
    normalize_y=False,
    random_state=42
)

# Fit on SCALED data
gpr.fit(X_scaled, y_scaled)

# Check diagnostics
print("\nGP Model Diagnostics:")
# The 'kernel_' attribute is set by the .fit() method after the GPR has been trained.
# If it's missing, it implies the fitting process either failed or was not completed.
if hasattr(gpr, 'kernel_'):
    print(f"  Kernel: {gpr.kernel_}")
    print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
else:
    print("  WARNING: 'kernel_' attribute not found. The Gaussian Process Regressor might not have been fitted successfully or the attribute was not assigned.")
    print(f"  Displaying initial kernel parameters instead:")
    print(f"  Initial Kernel: {gpr.kernel}")
    # Try to access length scales from the initial kernel if it has k2
    if hasattr(gpr.kernel, 'k2') and hasattr(gpr.kernel.k2, 'length_scale'):
        print(f"  Initial Length scales: {gpr.kernel.k2.length_scale}")
print(f"  Training R²: {gpr.score(X_scaled, y_scaled):.3f}")

# Check for overfitting
if gpr.score(X_scaled, y_scaled) > 0.99:
    print(" WARNING: Perfect fit (R²>0.99) suggests overfitting!")
    print("  Consider: more data, higher alpha, or simpler kernel")

# Define bounds for 6D space (use original scale)
bounds = [
    (X[:, 0].min(), X[:, 0].max()),
    (X[:, 1].min(), X[:, 1].max()),
    (X[:, 2].min(), X[:, 2].max()),
    (X[:, 3].min(), X[:, 3].max()),
    (X[:, 4].min(), X[:, 4].max()),
    (X[:, 5].min(), X[:, 5].max())  # Added the 6th dimension
]
print(f"\nBounds (original scale): {bounds}")

# Generate 6D candidates (in original scale)
sampler = qmc.LatinHypercube(d=6)  # Changed dimension to 6
X_candidates = qmc.scale(
    sampler.random(n=20000),  # More candidates for 6D
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
)

#Scale candidates before prediction
X_candidates_scaled = scaler_X.transform(X_candidates)

# Predict on SCALED candidates
y_pred_scaled, y_std_scaled = gpr.predict(X_candidates_scaled, return_std=True)

# Unscale predictions back to original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_std = y_std_scaled * scaler_y.scale_[0]  # Scale uncertainty

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]  # In original scale

print(f"\nNext Point to Sample (original scale):")
print(f"  X = {x_next}")
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}")
    print(f"      pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")

# Sanity checks
print(f"\nSanity Checks:")
print(f"  Predicted y range: [{y_pred.min():.3f}, {y_pred.max():.3f}]")
print(f"  Training y range: [{y.min():.3f}, {y.max():.3f}]")
print(f"  Uncertainty range: [{y_std.min():.3f}, {y_std.max():.3f}]")

if y_std.max() > 10 * np.abs(y.max() - y.min()):
    print("  WARNING: Uncertainty is way too high!")
    print("  This suggests the model is very uncertain everywhere")



Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

Training on 35 samples, 6D
Epoch 200/1000, Loss: 0.076093
Epoch 400/1000, Loss: 0.035688
Epoch 600/1000, Loss: 0.039707
Epoch 800/1000, Loss: 0.032148
Epoch 1000/1000, Loss: 0.039226

Training Results:
  MSE: 0.001383
  R²: 0.9904


In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 2D space
bounds = [
    (X[:, i].min(), X[:, i].max()) for i in range(n_dims)
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )


y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")

Bayessian Optimization

In [ ]:
class BayesianOptimizer:
    def __init__(self, bounds, n_initial=10, kernel=None, alpha=0.01, n_restarts=30):
        """
        Bayesian Optimization for hyperparameter tuning

        Args:
            bounds: List of (min, max) tuples for each dimension
            n_initial: Number of random initial points
            kernel: GP kernel (defaults to Matern)
            alpha: Noise regularization
            n_restarts: GP optimizer restarts
        """
        self.bounds = np.array(bounds)
        self.dim = len(bounds)
        self.n_initial = n_initial
        self.alpha = alpha
        self.n_restarts = n_restarts

        # Initialize scalers
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()

        # Setup kernel
        if kernel is None:
            self.kernel = ConstantKernel(1.0, constant_value_bounds=(1e-2, 1e3)) * \
                         Matern(nu=2.5,
                                length_scale=np.ones(self.dim),
                                length_scale_bounds=(0.01, 1000.0))
        else:
            self.kernel = kernel

        # Initialize GP
        self.gpr = GaussianProcessRegressor(
            kernel=self.kernel,
            n_restarts_optimizer=self.n_restarts,
            alpha=self.alpha,
            normalize_y=False,
            random_state=42
        )

        # Storage for observations - Initialize as lists and populate with initial data
        self.X_observed = X.tolist()  # Convert initial numpy array to list
        self.y_observed = y.tolist()  # Convert initial numpy array to list

        # Convert observed lists to numpy arrays for initial scaling and GP fitting
        X_initial_np = np.array(self.X_observed)
        y_initial_np = np.array(self.y_observed).reshape(-1, 1)

        # Fit scalers on the initial data provided
        self.scaler_X.fit(X_initial_np)
        self.scaler_y.fit(y_initial_np)

        # Fit GP on initial scaled data
        X_scaled_initial = self.scaler_X.transform(X_initial_np)
        y_scaled_initial = self.scaler_y.transform(y_initial_np).ravel()
        self.gpr.fit(X_scaled_initial, y_scaled_initial)

        # Print initial diagnostics
        r2 = self.gpr.score(X_scaled_initial, y_scaled_initial)
        print(f"Initial GP Model Fitted with {len(self.y_observed)} observations:")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print(" WARNING: Potential overfitting (R²>0.99) on initial data")

    def _get_initial_points(self):
        """Generate initial points using Latin Hypercube Sampling"""
        sampler = qmc.LatinHypercube(d=self.dim, seed=42)
        points = qmc.scale(
            sampler.random(n=self.n_initial),
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )
        return points

    def _acquisition_ucb(self, X, kappa=2.0):
        """Upper Confidence Bound acquisition function"""
        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
        y_std = y_std_scaled * self.scaler_y.scale_[0]

        # UCB (negate for minimization)
        return -(y_pred + kappa * y_std)[0]

    def _acquisition_ei(self, X, xi=0.01):
        """Expected Improvement acquisition function"""
        from scipy.stats import norm

        # Ensure scalers are fitted before transforming
        # This check is now redundant due to fitting in __init__, but kept for robustness
        if len(self.X_observed) == 0:
            raise RuntimeError("Scaler not fitted. Call update() with initial data first.")

        X_scaled = self.scaler_X.transform(X.reshape(1, -1))
        y_pred_scaled, y_std_scaled = self.gpr.predict(X_scaled, return_std=True)

        # Unscale
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()[0]
        y_std = y_std_scaled[0] * self.scaler_y.scale_[0]

        # Current best
        y_best = np.max(self.y_observed)

        # Avoid division by zero
        if y_std < 1e-10:
            return 0.0

        # Calculate EI
        z = (y_pred - y_best - xi) / y_std
        ei = (y_pred - y_best - xi) * norm.cdf(z) + y_std * norm.pdf(z)

        return -ei  # Negate for minimization

    def _propose_location(self, acquisition='ucb', kappa=2.0, xi=0.01, n_restarts=25):
        """Propose next sampling point by optimizing acquisition function"""

        # Choose acquisition function
        if acquisition == 'ucb':
            acq_func = lambda x: self._acquisition_ucb(x, kappa=kappa)
        elif acquisition == 'ei':
            acq_func = lambda x: self._acquisition_ei(x, xi=xi)
        else:
            raise ValueError(f"Unknown acquisition function: {acquisition}")

        # Multi-start optimization
        min_val = float('inf')
        min_x = None

        # Generate random starting points
        sampler = qmc.LatinHypercube(d=self.dim, seed=None)
        x0_samples = qmc.scale(
            sampler.random(n=n_restarts),
            l_bounds=self.bounds[:, 0],
            u_bounds=self.bounds[:, 1]
        )

        for x0 in x0_samples:
            result = minimize(
                acq_func,
                x0=x0,
                bounds=self.bounds,
                method='L-BFGS-B'
            )

            if result.fun < min_val:
                min_val = result.fun
                min_x = result.x

        return min_x

    def update(self, X_new, y_new):
        """Add new observations and refit GP"""
        # Ensure X_new is always treated as a 1D array for single point or a list of 1D arrays for multiple points
        if isinstance(X_new, np.ndarray) and X_new.ndim == 1:
            self.X_observed.append(X_new.tolist()) # Convert to list before appending to self.X_observed list
        elif isinstance(X_new, list) and all(isinstance(x, (list, np.ndarray)) for x in X_new):
            # If X_new is a list of lists/arrays, or a 2D numpy array
            for x_val in X_new:
                self.X_observed.append(x_val.tolist() if isinstance(x_val, np.ndarray) else x_val)
        else:
            # Assuming it's a single point that might not be a numpy array or list itself
            self.X_observed.append(X_new)

        if np.isscalar(y_new):
            self.y_observed.append(y_new)
        elif isinstance(y_new, (list, np.ndarray)):
            self.y_observed.extend(y_new)
        else:
            self.y_observed.append(y_new)

        # Convert observed lists to numpy arrays for scaling and GP fitting
        X_current = np.array(self.X_observed)
        y_current = np.array(self.y_observed).reshape(-1, 1)

        # Scale data - fit_transform on the growing dataset
        X_scaled = self.scaler_X.fit_transform(X_current)
        y_scaled = self.scaler_y.fit_transform(y_current).ravel()

        # Fit GP
        self.gpr.fit(X_scaled, y_scaled)

        # Print diagnostics
        r2 = self.gpr.score(X_scaled, y_scaled)
        print(f"\nGP Model Updated:")
        print(f"  Observations: {len(self.y_observed)}")
        print(f"  Training R²: {r2:.3f}")
        if hasattr(self.gpr, 'kernel_'):
            print(f"  Length scales: {self.gpr.kernel_.k2.length_scale}")
        if r2 > 0.99:
            print(" WARNING: Potential overfitting (R²>0.99)")

    def suggest(self, acquisition='ucb', kappa=2.0, xi=0.01):
        """Suggest next point to evaluate"""
        if len(self.y_observed) < self.n_initial:
            # Use random exploration initially
            # Ensure a distinct initial point is returned each time
            if len(self.X_observed) < self.n_initial:
                new_random_points = self._get_initial_points()
                # Find a point not already in self.X_observed (or similar logic)
                # For simplicity here, just return the next sequential initial point
                return new_random_points[len(self.X_observed)]
            else:
                # If we have enough observed points but not yet hit n_initial for proposal
                # This case is tricky if n_initial is smaller than initial X,y provided
                # Given the fix in __init__ this block will likely not be hit for new random points unless X_observed is explicitly cleared
                return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)
        else:
            # Use acquisition function
            return self._propose_location(acquisition=acquisition, kappa=kappa, xi=xi)

    def get_best(self):
        """Return best observed point"""
        if len(self.y_observed) == 0:
            return None, None

        best_idx = np.argmax(self.y_observed)
        return np.array(self.X_observed[best_idx]), self.y_observed[best_idx]


In [ ]:
# ==========================
# USAGE EXAMPLE
# ==========================

# Define your 8D bounds
bounds = [
    (0.0, 1.0),   # Parameter 1
    (0.0, 1.0),   # Parameter 2
    (0.0, 1.0),   # Parameter 3
    (0.0, 1.0),   # Parameter 4
    (0.0, 1.0),   # Parameter 5
    (0.0, 1.0),   # Parameter 6
]

# Initialize optimizer
optimizer = BayesianOptimizer(
    bounds=bounds,
    n_initial=20,      # Initial random samples
    alpha=0.01,        # GP noise
    n_restarts=30      # GP hyperparameter optimization restarts
)

# Define your objective function
def objective_function(params):
    """
    Your black-box function to optimize

    Args:
        params: Array of 6 parameters
    Returns:
        score: Float (higher is better)
    """
    # Example: replace with your actual function
    # e.g., train model with these hyperparameters and return validation score
    score = -np.sum((params - 0.5)**2)  # Dummy function
    return score

# Bayesian Optimization Loop
n_iterations = 10

print("Starting Bayesian Optimization...\n")

for i in range(n_iterations):
    # Get suggestion
    x_next = optimizer.suggest(acquisition='ucb', kappa=2.0)

    # Evaluate objective
    y_next = objective_function(x_next)

    # Update optimizer
    optimizer.update(x_next, y_next)

    # Get current best
    x_best, y_best = optimizer.get_best()

    print(f"Iteration {i+1}/{n_iterations}")
    print(f"  Suggested: {x_next}")
    print(f"  Score: {y_next:.4f}")
    print(f"  Best so far: {y_best:.4f}")
    print(f"  Best params: {x_best}\n")

# Final result
x_best, y_best = optimizer.get_best()
print("\n" + "="*50)
print("OPTIMIZATION COMPLETE")
print("="*50)
print(f"Best score: {y_best:.4f}")
print(f"Best parameters: {x_best}")